# ⚙️ Notebook 2: ChatGPT — Serving & Scaling


## 🛠️ Setup

```bash
cd 06-system-designs/chatgpt
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

Everything here is a **deterministic discrete-event simulation** in pure Python. Seeded, so
the numbers are the same every run. Run cells top-to-bottom.


## 💡 The serving path

Notebook 1 sized the fleet. This notebook builds the thing that decides *which request runs
on which GPU, when* — and that scheduler is where almost all the engineering value of an
inference stack lives.

```
  request ──▶ [ bounded queue ] ──▶ [ scheduler ] ──▶ [ replica: GPU(s) + KV cache ]
                    │                     │                        │
                 shed if                admit if              stream tokens out
                 too deep            KV memory fits
```

Four questions, in the order they bite:

1. **How do we run more than one request on a GPU at a time?** (batching)
2. **Why do the two halves of a request — reading the prompt and writing the reply — behave
   like different workloads?** (prefill vs decode)
3. **What happens when more work arrives than the fleet can do?** (admission control)
4. **What is the actual capacity limit?** (KV cache memory — not FLOPs)

Throughout, one **replica** = one model instance, which in notebook 1's terms is a
tensor-parallel group of ~4 GPUs. Everything is measured per replica; multiply by the fleet
size to get aggregate numbers.


## 🔩 The cost model

Before simulating we need a model of how long GPU work takes. Two operations, two completely
different cost curves — and getting this right is most of the insight.

**Decode** (generate one token for every sequence in the batch): the GPU must read the entire
model's weights out of HBM to compute one step, *regardless of how many sequences are in the
batch*. So a decode step costs a large fixed price plus a small per-sequence price:

```
decode_step_seconds = WEIGHT_STREAM_S + PER_SEQ_S × batch_size
```

**Prefill** (process a whole prompt at once): every token in the prompt is a row in a big
matrix multiply. The GPU is compute-saturated, and cost is simply proportional to tokens:

```
prefill_seconds = prompt_tokens / PREFILL_TOKENS_PER_SEC
```

That asymmetry — decode has a huge fixed cost you can amortise, prefill does not — is the
entire reason batching works, and the reason it only helps one of the two phases.


In [ ]:
# ==== ASSUMPTIONS: one serving replica (~4 GPUs, tensor-parallel) ====
# Plausible orders of magnitude for a mid-size dense model. Not measurements.
WEIGHT_STREAM_S  = 0.015      # fixed cost per decode step: stream weights out of HBM
                              # [range: 0.005 - 0.05 s]
PER_SEQ_S        = 0.00005    # marginal decode cost per sequence in the batch
PREFILL_TOK_S    = 20_000     # prefill throughput for the WHOLE replica, tokens/s
                              # [range: 5k - 100k]. Deliberately more conservative than
                              # notebook 1's per-GPU figure, because that one ignored
                              # attention -- which is what actually dominates long prompts.
KV_TOKENS        = 60_000     # KV-cache capacity of this replica, in tokens
                              # (notebook 1: ~40 GB free HBM/GPU ÷ 256 KiB/token × 4 GPUs)
MAX_BATCH        = 256        # scheduler's hard cap on concurrent sequences
MAX_PREFILLS_PER_STEP = 4     # how many new prompts we'll process between decode steps

def decode_step_seconds(batch_size: int) -> float:
    """Time for ONE decode step that advances every sequence in the batch by one token."""
    return WEIGHT_STREAM_S + PER_SEQ_S * batch_size

def prefill_seconds(prompt_tokens: int) -> float:
    """Time to process a whole prompt and produce the first token."""
    return prompt_tokens / PREFILL_TOK_S

print(f'{"batch":>6} | {"step (ms)":>10} | {"tok/s (total)":>14} | {"tok/s per user":>15}')
print('-' * 56)
for b in (1, 2, 8, 32, 64, 128, 256):
    st = decode_step_seconds(b)
    print(f'{b:>6} | {st*1000:>10.2f} | {b/st:>14,.0f} | {1/st:>15,.1f}')

print()
print('Read the last two columns together. Going from batch 1 to batch 128 multiplies')
print(f'replica throughput by {(128/decode_step_seconds(128))/(1/decode_step_seconds(1)):.0f}x while making each individual user only')
print(f'{(1/decode_step_seconds(1))/(1/decode_step_seconds(128)):.2f}x slower. That is the best trade in the entire system, and it is')
print('why an unbatched inference server is not a serious inference server.')


### Why the fixed cost exists: arithmetic intensity

This isn't a magic constant — it falls out of a two-line calculation you can do for any
accelerator, and it tells you exactly what batch size you need.

For each decoded token per sequence, the GPU does about `2 × params` FLOPs (one multiply-add
per weight). To do that it must read `bytes_per_param × params` bytes of weights from HBM —
**once for the whole batch**. So:

```
arithmetic intensity = FLOPs / bytes
                     = (2 · P · batch) / (bytes_per_param · P)
                     = 2 · batch / bytes_per_param
```

For bf16 weights that is exactly **`batch` FLOPs per byte** — the batch size *is* the
arithmetic intensity. Every accelerator has a *ridge point*, `peak_FLOPS / HBM_bandwidth`:
below it you are memory-bound (the arithmetic units idle waiting for data), above it you are
compute-bound.


In [ ]:
# ==== ASSUMPTIONS: accelerator (per replica, aggregated over its GPUs) ====
PEAK_TFLOPS       = 1_600.0   # bf16 dense, effective, across the replica  [range: 400 - 8000]
HBM_BANDWIDTH_TBS = 12.0      # TB/s across the replica                    [range: 3 - 30]
BYTES_PER_PARAM   = 2

ridge_point = (PEAK_TFLOPS * 1e12) / (HBM_BANDWIDTH_TBS * 1e12)   # FLOPs per byte
print(f'Accelerator ridge point: {ridge_point:,.0f} FLOPs/byte')
print('(above this the chip is compute-bound; below it, memory-bandwidth-bound)')
print()
print(f'{"phase":>22} | {"arithmetic intensity":>21} | bound by')
print('-' * 62)
for b in (1, 8, 64, 256, 1024):
    ai = b / BYTES_PER_PARAM * 2      # (2*P*b FLOPs) / (BYTES_PER_PARAM*P bytes)
    print(f'{"decode, batch=%d" % b:>22} | {ai:>21,.0f} | '
          f'{"COMPUTE" if ai >= ridge_point else "memory bandwidth"}')
for p in (512, 4096):
    ai = p / BYTES_PER_PARAM * 2      # a prompt of p tokens is p rows through the same weights
    print(f'{"prefill, %d tokens" % p:>22} | {ai:>21,.0f} | '
          f'{"COMPUTE" if ai >= ridge_point else "memory bandwidth"}')

print()
print(f'So: decode needs a batch of ~{ridge_point*BYTES_PER_PARAM/2:,.0f} sequences before the GPU stops idling,')
print('while a single 512-token prompt is already compute-bound on its own.')
print()
print('This is the whole prefill/decode story in one table:')
print('  - PREFILL  is compute-bound. Batching it buys you almost nothing; the chip is')
print('             already busy. Long prompts are just expensive, full stop.')
print('  - DECODE   is memory-bandwidth-bound at any realistic batch size. Batching is the')
print('             ONLY thing that helps, and it helps enormously.')
print()
print('It also explains a result that surprises people: a 4,000-token prompt costs about the')
print(f'same wall-clock as {4000/PREFILL_TOK_S/decode_step_seconds(64):.0f} decode steps — reading is cheap, writing is not.')


⚖️ **What this cost model is hiding:**

- **Attention is not in it.** We charged only for weight streaming. Attention cost grows with
  *context length*, so a batch of 128 conversations at 100k tokens each behaves nothing like
  a batch of 128 at 1k tokens. Our model will therefore be optimistic for long contexts.
- **`PER_SEQ_S` being tiny is an idealisation.** Real kernels have ragged-batch overheads,
  and past some batch size you start spilling out of on-chip memory and the curve bends.
- **No pipeline bubbles, no network.** Tensor-parallel replicas synchronise every layer; that
  cost is folded into `WEIGHT_STREAM_S` here rather than modelled.

These all make the simulation *kinder* than reality, which is the right direction: if a design
fails here, it definitely fails in production.


## 🧪 The workload

A seeded set of requests: Poisson arrivals, log-normal prompt and output lengths (both are
heavy-tailed in reality, which matters enormously for batching).


In [ ]:
import random, statistics
from collections import deque

class Req:
    """One inference request moving through the system."""
    __slots__ = ('i', 'arrival', 'prompt', 'output', 'desired',
                 'first_token_at', 'done_at', 'rejected')
    def __init__(self, i, arrival, prompt, output, desired):
        self.i, self.arrival, self.prompt, self.output = i, arrival, prompt, output
        self.desired = desired          # what the user WANTED, before max_tokens clipped it
        self.first_token_at = None
        self.done_at = None
        self.rejected = False

# ==== ASSUMPTIONS: request shape ====
MAX_TOKENS       = 4_096   # the output cap we advertise to clients [range: 1k - 128k]
LONG_REPLY_FRAC  = 0.05    # share of turns that are long-form ("write me an essay")

def make_workload(n: int, rate_per_sec: float, seed: int = 7):
    """Poisson arrivals; heavy-tailed prompt/output lengths. Deterministic for a given seed.

    Output length is a MIXTURE: most turns are short answers, a minority are long-form.
    That mixture is not decoration -- a single fat tail is what breaks static batching
    and what makes draining a replica slow. A pure log-normal is too polite.
    """
    rng = random.Random(seed)
    t = 0.0
    out = []
    for i in range(n):
        t += rng.expovariate(rate_per_sec)
        prompt = int(rng.lognormvariate(6.0, 0.6))         # median ~403 tokens
        if rng.random() < LONG_REPLY_FRAC:
            output = int(rng.lognormvariate(7.4, 0.5))     # long-form, median ~1,600
        else:
            output = int(rng.lognormvariate(5.3, 0.7))     # short answer, median ~200
        desired = max(8, output)
        out.append(Req(i, t, max(32, min(prompt, 4_000)),
                       min(desired, MAX_TOKENS), desired))
    return out

_w = make_workload(4_000, 10.0)
_p = sorted(r.prompt for r in _w)
_o = sorted(r.output for r in _w)
pct = lambda a, q: a[min(len(a) - 1, int(q * len(a)))]
print(f'{"":8} {"p50":>7} {"p90":>7} {"p99":>7} {"max":>7} {"mean":>7}')
print(f'{"prompt":8} {pct(_p,.5):>7,} {pct(_p,.9):>7,} {pct(_p,.99):>7,} {max(_p):>7,} '
      f'{statistics.mean(_p):>7,.0f}')
print(f'{"output":8} {pct(_o,.5):>7,} {pct(_o,.9):>7,} {pct(_o,.99):>7,} {max(_o):>7,} '
      f'{statistics.mean(_o):>7,.0f}')
print()
print(f'The p99 output is {pct(_o,.99)/pct(_o,.5):.1f}x the median. Hold on to that ratio — it is the reason')
print('static batching fails, and the reason draining a replica before a deploy is slow.')
print('Note it is the TAIL that does the damage, not the mean. Two workloads with identical')
print('mean output length can differ 5x in achievable throughput.')


In [ ]:
# ==== Metrics helper, shared by every strategy below ====
def report(name, reqs, extra=''):
    served = [r for r in reqs if not r.rejected and r.done_at is not None]
    rejected = sum(1 for r in reqs if r.rejected)
    if not served:
        print(f'{name:<24} nothing completed'); return None
    e2e  = sorted(r.done_at - r.arrival for r in served)
    ttft = sorted(r.first_token_at - r.arrival for r in served)
    makespan = max(r.done_at for r in served) - min(r.arrival for r in reqs)
    q = lambda a, p: a[min(len(a) - 1, int(p * len(a)))]
    m = dict(throughput=len(served) / makespan,
             ttft_p50=q(ttft, .50), ttft_p95=q(ttft, .95),
             e2e_p50=q(e2e, .50),  e2e_p95=q(e2e, .95),
             served=len(served), rejected=rejected)
    print(f'{name:<24} thr={m["throughput"]:>6.2f} req/s | '
          f'TTFT p50={m["ttft_p50"]:>7.2f}s p95={m["ttft_p95"]:>7.2f}s | '
          f'e2e p50={m["e2e_p50"]:>6.1f}s p95={m["e2e_p95"]:>6.1f}s{extra}')
    return m


## ❌ BAD — one request at a time per replica

The obvious design, and the one you get for free if you wrap a model in a Flask handler: a
request arrives, the GPU works on it until the reply is finished, then it takes the next one.


In [ ]:
def sim_naive(reqs):
    """One request occupies the whole replica from prefill to last token."""
    clock = 0.0
    for r in reqs:
        clock = max(clock, r.arrival)                  # idle until something arrives
        clock += prefill_seconds(r.prompt)
        r.first_token_at = clock
        clock += r.output * decode_step_seconds(1)     # batch of exactly 1, every step
        r.done_at = clock
    return reqs

for rate in (0.1, 0.25, 1.0, 5.0):
    report(f'naive  @ {rate:>4} req/s', sim_naive(make_workload(300, rate)))

print()
naive_ceiling = report('naive  @ SATURATION', sim_naive(make_workload(300, 50.0)))['throughput']
print()
print(f'Ceiling: ~{naive_ceiling:.2f} requests/second per replica.')
print('Offer it more than that and latency simply grows without limit — throughput does not')
print('move, because the GPU is already 100% busy doing the least efficient thing it can do:')
print(f'batch-of-1 decode, running at {1/decode_step_seconds(1):.0f} tokens/s while the hardware could do '
      f'{256/decode_step_seconds(256):,.0f}.')


**One request per GPU wastes ~98% of the machine.** Not because the code is slow, but because
decode at batch 1 has an arithmetic intensity of 2 FLOPs/byte against a ridge point in the
hundreds. The arithmetic units are idle almost all the time, waiting for weights.

## 🙂 BETTER — static batching

So batch. Collect requests until you have `B` of them, run them together, return the results,
take the next `B`.

But how big can `B` be? Not as big as you'd like, and the reason is memory. Every sequence in
the batch needs KV space for its prompt *plus* every token it might generate — and since you
don't know how many it will generate, a static scheduler has to reserve `max_tokens` for each.

This produces one of the least obvious couplings in the whole design: **the maximum output
length you advertise to users directly sets your batch size, and therefore your throughput.**


In [ ]:
avg_prompt = statistics.mean(r.prompt for r in _w)

def static_batch_limit(max_tokens):
    return max(1, int(KV_TOKENS / (avg_prompt + max_tokens)))

print(f'KV capacity: {KV_TOKENS:,} tokens.   Average prompt: {avg_prompt:,.0f} tokens.\n')
print(f'{"advertised max_tokens":>21} | {"reserved/slot":>13} | {"largest static batch":>20}')
print('-' * 60)
for mt in (512, 1_024, 4_096, 16_384):
    print(f'{mt:>21,} | {avg_prompt+mt:>13,.0f} | {static_batch_limit(mt):>20}')

B_STATIC = static_batch_limit(MAX_TOKENS)
print()
print(f'We advertise max_tokens={MAX_TOKENS:,}, so memory gives us a batch of {B_STATIC} — '
      f'against a\nMAX_BATCH of {MAX_BATCH} we would happily have used.')
print(f'The median reply is {pct(_o,.5):,} tokens, so {(1 - pct(_o,.5)/MAX_TOKENS):.0%} of every reservation sits empty,')
print('booked against a long answer that almost never arrives.')


In [ ]:
def sim_static(reqs, batch_size=32):
    """Fill a batch, prefill it, decode until the LONGEST sequence finishes, repeat.

    The batch slot is held for the whole batch even after that sequence has finished --
    that is what makes it 'static', and what makes it slow.
    """
    clock = 0.0
    i, n = 0, len(reqs)
    while i < n:
        if reqs[i].arrival > clock:
            clock = reqs[i].arrival                    # wait for the first arrival
        batch = []
        while i < n and len(batch) < batch_size and reqs[i].arrival <= clock:
            batch.append(reqs[i]); i += 1
        if not batch:                                  # queue empty: take one and go
            batch.append(reqs[i]); i += 1
        for r in batch:                                # prefill each prompt
            clock += prefill_seconds(r.prompt)
            r.first_token_at = clock
        # Steps are padded to the full batch width: finished slots still cost compute.
        step = decode_step_seconds(batch_size)
        longest = max(r.output for r in batch)
        for k in range(1, longest + 1):
            clock += step
            for r in batch:
                if r.output == k:
                    r.done_at = clock
    return reqs

for rate in (0.2, 0.5, 1.0, 5.0):
    report(f'static B={B_STATIC} @ {rate:>4} req/s',
           sim_static(make_workload(400, rate), B_STATIC))
print()
static_ceiling = report(f'static B={B_STATIC} @ SATURATION',
                        sim_static(make_workload(400, 50.0), B_STATIC))['throughput']
print()
print(f'Ceiling: {static_ceiling:.2f} req/s — {static_ceiling/naive_ceiling:.1f}x better than naive. Real progress.')
print()
print('But look at TTFT under load. It is not tens of milliseconds, it is minutes.')
print('Two reasons, and both are structural:')
print('  1. HEAD-OF-LINE BLOCKING AT THE START: you arrive just after a batch launched, so')
print('     you wait for the whole batch to finish before you are even looked at.')
print('  2. HEAD-OF-LINE BLOCKING AT THE END: the batch runs until its LONGEST member is')
print(f'     done. The p99 output is {pct(_o,.99):,} tokens against a median of {pct(_o,.5):,}, so ONE')
print(f'     unlucky long reply pins the other {B_STATIC-1} slots — finished, still computed as padding.')


In [ ]:
# ==== The product decision hiding inside the scheduler ====
# Cut the advertised max_tokens and you can afford a bigger batch. What does that buy?
print(f'{"max_tokens":>11} | {"batch":>6} | {"ceiling req/s":>14} | cost to the user')
print('-' * 68)
for mt in (16_384, 4_096, 1_024, 512):
    B = static_batch_limit(mt)
    truncated = sum(1 for r in _w if r.desired > mt) / len(_w)
    rs = sim_static(make_workload(400, 50.0), B)
    served = [r for r in rs if r.done_at is not None]
    thr = len(served) / (max(r.done_at for r in served) - min(r.arrival for r in rs))
    print(f'{mt:>11,} | {B:>6} | {thr:>14.2f} | {truncated:>5.1%} of replies cut off')

print()
print('With static batching, "how long may an answer be?" and "how many users can we serve?"')
print('are the same question. That is a terrible place for a product decision to live, and')
print('continuous batching exists partly to separate them again.')


In [ ]:
# ==== How much of the static batch is actually doing useful work? ====
rng = random.Random(11)
sample = [r.output for r in make_workload(B_STATIC, 10.0, seed=11)]
longest = max(sample)
useful = sum(sample)
paid_for = longest * len(sample)
print(f'A batch of {len(sample)} with these output lengths runs for {longest} steps '
      f'(median member: {sorted(sample)[len(sample)//2]}).')
print(f'  slot-steps of real work : {useful:>8,}')
print(f'  slot-steps paid for     : {paid_for:>8,}')
print(f'  wasted on padding       : {1 - useful/paid_for:>8.0%}')
print()
print(f'{1 - useful/paid_for:.0%} of the GPU, burned computing padding for sequences that already ended.')
print('Bigger batches make this WORSE, not better: the maximum of more samples is larger,')
print('so every extra slot you add is held hostage by an even longer straggler.')


## ✅ BEST — continuous batching

The fix is to stop thinking in batches and start thinking in **slots**.

The scheduler keeps a set of active sequences. Every decode step advances all of them by one
token. The instant a sequence emits its end-of-sequence token, **its slot is freed and a
waiting request is admitted into it immediately** — no waiting for batch boundaries, no
padding, no head-of-line blocking.

```
static batching                    continuous batching
step: A B C D                      step: A B C D
      A B C D                            A B C D
      A B . D    <- C done, slot idle          A B E D    <- E admitted into C's slot
      A B . D                            A B E D
      A . . D    <- padding waste              A F E D    <- F admitted into B's slot
      A . . .                            G F E D
```

Two details that make this work in practice and that most explanations skip:

- **Prefill blocks decode.** A newly-admitted request's prompt must be processed before it can
  join the decode loop, and that occupies the same GPU. Admitting a burst of long prompts
  *stalls everyone else's token stream*. Hence `MAX_PREFILLS_PER_STEP` — a deliberate cap that
  trades TTFT for smooth inter-token latency. (Production engines go further and chop a long
  prefill into pieces interleaved with decode steps: "chunked prefill".)
- **Admission is a memory allocation.** A request only enters if the KV cache has room for it.
  This is the real capacity limit, and we make it explicit below.


In [ ]:
def sim_continuous(reqs, max_batch=MAX_BATCH, kv_tokens=KV_TOKENS,
                   queue_limit=None, collect_batch_sizes=None):
    """Continuous batching: slots are freed and refilled every step.

    Admission reserves prompt + the request's actual output length. That is a deliberate
    idealisation -- a real scheduler cannot know the output length in advance. Section 6
    removes the cheat and measures what it costs.
    """
    clock = 0.0
    i, n = 0, len(reqs)
    active = []            # [req, tokens_remaining, kv_held]
    waiting = deque()
    kv_used = 0

    while i < n or waiting or active:
        while i < n and reqs[i].arrival <= clock:            # new arrivals
            r = reqs[i]; i += 1
            if queue_limit is not None and len(waiting) >= queue_limit:
                r.rejected = True                            # fast 429, no GPU spent
                r.first_token_at = r.done_at = r.arrival
            else:
                waiting.append(r)

        if not active and not waiting:                       # idle: jump to next arrival
            if i < n: clock = reqs[i].arrival; continue
            break

        admitted = 0                                         # --- admit into free slots ---
        while waiting and admitted < MAX_PREFILLS_PER_STEP and len(active) < max_batch:
            r = waiting[0]
            if kv_used + r.prompt + r.output > kv_tokens:     # no KV room: stop admitting
                break
            waiting.popleft(); admitted += 1
            clock += prefill_seconds(r.prompt)               # prefill occupies the GPU
            r.first_token_at = clock
            kv_used += r.prompt
            active.append([r, r.output, r.prompt])

        if not active:                                       # blocked on KV, nothing running
            clock = max(clock, reqs[i].arrival) if i < n else clock + 0.001
            continue

        if collect_batch_sizes is not None:
            collect_batch_sizes.append(len(active))
        clock += decode_step_seconds(len(active))            # --- one decode step ---
        still = []
        for e in active:
            e[1] -= 1; e[2] += 1; kv_used += 1
            if e[1] <= 0:
                e[0].done_at = clock                         # finished: free the slot NOW
                kv_used -= e[2]
            else:
                still.append(e)
        active = still
    return reqs

batches = []
for rate in (0.5, 1.0, 2.0, 10.0):
    report(f'continuous @ {rate:>4} req/s', sim_continuous(make_workload(400, rate)))
print()
cont = sim_continuous(make_workload(1200, 50.0), collect_batch_sizes=batches)
cont_ceiling = report('continuous @ SATURATION', cont)['throughput']
print()
print(f'Average batch size while saturated: {statistics.mean(batches):.0f} '
      f'(cap was MAX_BATCH={MAX_BATCH})')
print(f'The ridge point said we need ~{ridge_point*BYTES_PER_PARAM/2:.0f} for the GPU to be compute-bound, so even fully')
print('loaded this replica is still memory-bandwidth-bound. KV capacity, not the batch cap,')
print('is what stopped us — which is the subject of the last section.')


In [ ]:
# ==== Head to head ====
print(f'{"strategy":<26} {"ceiling req/s":>14} {"vs naive":>10} '
      f'{"TTFT p50 @ 2 req/s":>20}')
print('(2 req/s is ~8x the naive ceiling, so its TTFT there is queue wait, not service time)')
print('-' * 74)
rows = [
    ('naive (1 req / replica)', naive_ceiling, sim_naive(make_workload(300, 2.0))),
    (f'static batching B={B_STATIC}',  static_ceiling, sim_static(make_workload(400, 2.0), B_STATIC)),
    ('continuous batching',     cont_ceiling,   sim_continuous(make_workload(400, 2.0))),
]
for name, ceiling, r2 in rows:
    ttft = sorted(x.first_token_at - x.arrival for x in r2 if x.first_token_at is not None)
    print(f'{name:<26} {ceiling:>14.2f} {ceiling/naive_ceiling:>9.0f}x '
          f'{ttft[len(ttft)//2]:>19.2f}s')

print()
print('Continuous batching is not a tweak. It is the difference between needing')
print(f'{cont_ceiling/naive_ceiling:.0f} replicas and needing 1.')
print()
print('⚖️  What it costs you:')
print('  - Per-user speed drops. At batch ~100 each user gets '
      f'{1/decode_step_seconds(100):.0f} tok/s instead of the')
print(f'    {1/decode_step_seconds(1):.0f} tok/s a dedicated GPU would give, and at the {MAX_BATCH} cap it is '
      f'{1/decode_step_seconds(MAX_BATCH):.0f} tok/s.')
print('    You are selling per-user latency to buy throughput. That trade is fine only')
print('    while you stay above reading speed (~250 wpm ≈ 5 tok/s); below it, users watch')
print('    the text crawl and the batching win turns into a product regression.')
print('  - Enormous scheduler complexity: paged KV allocation, preemption, per-step')
print('    admission. This is why teams adopt vLLM/TensorRT-LLM/SGLang rather than write it.')
print('  - Noisy neighbours become real. One user\'s 100k-token prompt stalls the decode')
print('    loop for everyone on that replica during its prefill.')
print('  - Per-request performance is no longer predictable or reproducible: your latency')
print('    now depends on who else happens to be on your replica.')


## 📡 Streaming tokens back

The reply is produced one token at a time over many seconds, so we push tokens to the client
as they appear rather than making them wait for the whole thing.

| | Server-Sent Events (SSE) | WebSocket |
|---|---|---|
| Direction | server → client only | bidirectional |
| Transport | plain HTTP response, `text/event-stream` | HTTP upgrade, then framed |
| Proxies / CDNs | works with ordinary HTTP infra | needs upgrade support end to end |
| Reconnect + resume | built in: `Last-Event-ID` header | you build it yourself |
| Good for | streaming a completion | live voice, interruption, multiplexed sessions |

**SSE is the right default** for text chat, precisely because resume is in the protocol and
because an HTTP response survives every proxy on earth. Reach for WebSocket when the client
needs to talk *during* the generation — interrupting, or streaming audio in.


In [ ]:
# ==== A minimal SSE encoder: note the sequence id on every frame ====
def sse_frames(tokens, heartbeat_every=8):
    """Yield SSE wire frames. The `id:` is what makes resume possible (notebook 3)."""
    for seq, tok in enumerate(tokens):
        if seq and seq % heartbeat_every == 0:
            yield ': keep-alive\n\n'          # comment frame: bytes on the wire, no content
        yield f'id: {seq}\nevent: token\ndata: {{"t": {tok!r}}}\n\n'
    yield f'id: {len(tokens)}\nevent: done\ndata: {{"finish": "stop"}}\n\n'

demo = ['Continuous', ' batching', ' frees', ' the', ' slot', ' immediately', '.']
for frame in list(sse_frames(demo, heartbeat_every=4))[:6]:
    print(repr(frame))
print('...')
print()
print('Three things are load-bearing in those seven lines:')
print('  - `id:` gives every token a sequence number, so a reconnecting client can say')
print('    "I have up to 42" via Last-Event-ID and get only the tail.')
print('  - the keep-alive comment frame puts bytes on the wire during long pauses, so idle')
print('    timeouts in proxies do not kill a healthy stream.')
print('  - there is no Content-Length and no way to add one. Everything downstream must')
print('    forward bytes as they arrive; one buffering proxy in the path silently converts')
print('    your streaming API into a slow non-streaming one.')


In [ ]:
# ==== What streaming does to timeouts ====
# A conventional proxy idle-timeout fires if NO bytes flow for N seconds. Under load,
# TTFT is the gap before the first byte -- so the timeout is really a TTFT budget.
PROXY_IDLE_TIMEOUT_S = 30.0

print(f'{"offered load":>13} | {"TTFT p50":>9} | {"TTFT p95":>9} | '
      f'{"killed by %ds idle timeout" % PROXY_IDLE_TIMEOUT_S:>28}')
print('-' * 72)
for rate in (2.0, 10.0, 15.0, 25.0, 40.0):
    rs = sim_continuous(make_workload(2_000, rate))
    ttft = sorted(r.first_token_at - r.arrival for r in rs if r.first_token_at is not None)
    killed = sum(1 for x in ttft if x > PROXY_IDLE_TIMEOUT_S) / len(ttft)
    q = lambda a, p: a[min(len(a) - 1, int(p * len(a)))]
    print(f'{rate:>10.0f}/s | {q(ttft,.5):>8.2f}s | {q(ttft,.95):>8.2f}s | {killed:>27.1%}')

print()
print('The failure this produces is uniquely nasty: the client gets a *successful* HTTP')
print('response that simply stops. No error body -- the 200 OK went out with the first')
print('token, long before anything went wrong. You cannot retroactively send a 503.')
print()
print('So the rules for a streaming path are:')
print('  1. Set the connection idle timeout ABOVE your worst tolerable TTFT, then defend')
print('     that TTFT with admission control (next section) rather than with a long timeout.')
print('  2. Send heartbeat frames so a long generation never looks idle.')
print('  3. Put the real deadline where it belongs: a per-request generation budget the')
print('     server enforces, ending the stream with an explicit `event: error` frame it')
print('     still has a live connection to send.')
print('  4. Never let a load balancer retry a streaming request. Once one byte is out,')
print('     retrying means the user sees two half-answers.')


In [ ]:
# ==== Rolling deploys: you cannot drain a replica quickly ====
# Stop admitting, let the in-flight generations finish. How long is that?
K8S_DEFAULT_GRACE_S = 30.0

def drain_time(output_cap, n_active=100, seed=3):
    remaining = sorted(min(r.desired, output_cap)
                       for r in make_workload(n_active, 10.0, seed=seed))
    t, steps, finished_at = 0.0, 0, []
    while remaining:
        t += decode_step_seconds(len(remaining))
        steps += 1
        while remaining and remaining[0] <= steps:
            remaining.pop(0); finished_at.append(t)
    return finished_at

print(f'Draining a replica with 100 in-flight generations, by advertised max_tokens:')
print(f'{"max_tokens":>11} | {"p50 done":>9} | {"p95 done":>9} | {"all done":>9} | '
      f'fits in {K8S_DEFAULT_GRACE_S:.0f}s grace?')
print('-' * 70)
for cap in (256, 512, 1_024, 4_096):
    f = drain_time(cap)
    print(f'{cap:>11,} | {f[49]:>8.1f}s | {f[94]:>8.1f}s | {f[-1]:>8.1f}s | '
          f'{"yes" if f[-1] <= K8S_DEFAULT_GRACE_S else "NO - truncated mid-sentence"}')

print()
print(f'(Above ~4k the cap stops binding: with this workload almost nothing wants more,')
print(f' so raising max_tokens past {MAX_TOKENS:,} changes drain time not at all — until the day')
print(' someone ships a feature that generates long documents.)')
print()
print('Drain time is set by your longest permitted generation, not by your average one.')
print('Raise max_tokens for a nicer product and you have silently lengthened every deploy.')
print('The tail is also self-worsening in an interesting way: as the replica drains the')
print('batch shrinks, so each remaining user speeds UP -- but the last sequence still has')
print('to emit every one of its tokens, and nothing can make that shorter.')
print()
print('Options, all imperfect:')
print('  - long termination grace periods -> slow, risky deploys')
print('  - migrate the KV cache to another replica -> complex, and ~256 KiB/token to copy')
print('  - accept truncation for the unlucky tail and give them a retry affordance')
print('  - drain by stopping admission well before the rollout, at the cost of capacity')


## 🚦 Admission control and load shedding

A queue in front of a fixed-capacity service is not a buffer. It is a **latency amplifier**.

If arrivals exceed capacity, queue depth grows linearly with time and so does waiting time.
Users give up and retry — which adds *more* load. Meanwhile the GPU keeps dutifully generating
tokens for requests whose owners left minutes ago.

The counterintuitive claim we are about to demonstrate: **a server that rejects a third of its
traffic answers more users than one that accepts everything.**


In [ ]:
CLIENT_TIMEOUT_S = 60.0     # after this the user has closed the tab; the answer is worthless

def admission_experiment(queue_limit, rate=25.0, n=2_000):
    reqs = sim_continuous(make_workload(n, rate), queue_limit=queue_limit)
    rejected = [r for r in reqs if r.rejected]
    served   = [r for r in reqs if not r.rejected]
    in_time  = [r for r in served if r.done_at - r.arrival <= CLIENT_TIMEOUT_S]
    too_late = [r for r in served if r.done_at - r.arrival >  CLIENT_TIMEOUT_S]
    lat = sorted(r.done_at - r.arrival for r in in_time)
    span = max(r.done_at for r in served) - min(r.arrival for r in reqs)
    wasted = sum(r.output for r in too_late) / max(sum(r.output for r in served), 1)
    q = lambda a, p: a[min(len(a) - 1, int(p * len(a)))] if a else float('nan')
    label = 'unbounded queue' if queue_limit is None else f'queue<={queue_limit} + fast 429'
    print(f'{label:<24} rejected={len(rejected):>4} answered_in_time={len(in_time):>4} '
          f'timed_out={len(too_late):>4} | goodput={len(in_time)/span:>5.2f}/s '
          f'p50={q(lat,.5):>5.1f}s p95={q(lat,.95):>5.1f}s | GPU wasted={wasted:>5.1%}')

print(f'Offered load 25 req/s against a replica whose ceiling is ~{cont_ceiling:.0f} req/s.')
print(f'Client gives up after {CLIENT_TIMEOUT_S:.0f}s.\n')
admission_experiment(None)
for limit in (400, 100, 40):
    admission_experiment(limit)


Read those rows carefully, because the result is genuinely counterintuitive.

- The **unbounded queue rejects nobody** — and roughly **29% of users still get nothing**,
  because their answer arrives after they have gone. Worse, a third of all GPU time went into
  producing tokens for closed browser tabs. That capacity was not merely idle; it was
  *actively spent on waste*, which is why goodput drops below the ceiling we measured earlier.
- The **bounded queue rejects several hundred users instantly** — and answers **more** users
  than the unbounded one, at a better p95. Rejecting fast is not a lesser outcome. It
  *creates* capacity, by refusing to spend it on work that is already doomed.
- Tightening the limit further trades reach for experience: fewer people served, but those who
  are served get a p50 of a few seconds instead of half a minute.

**The queue limit is a product decision expressed as a number.** "How long may a user wait
before we would rather tell them no?" Divide that by service time and you have your limit.

⚖️ **What load shedding costs you:**

- **A 429 is a real failure for that user.** You have chosen who loses. Do it deliberately:
  shed free-tier before paid, shed retries before first attempts, shed a regenerate before a
  first message.
- **It invites retry storms.** A rejected client that immediately retries converts one
  rejection into ten. `Retry-After` plus client-side exponential backoff with jitter is not
  optional — without it, load shedding makes overload worse.
- **The limit needs to adapt.** A fixed depth is wrong the moment request mix changes. Better
  controllers watch measured wait time and shed to hold a latency target (CoDel-style), or
  size the queue from live capacity estimates.
- **Cancellation must propagate.** Everything above assumes we stop generating when the client
  disconnects. If the disconnect does not reach the scheduler, the waste comes back in full.


## 🧠 The KV cache is the real capacity limit

Back to the constraint from notebook 1. Every active sequence holds KV memory proportional to
its context length, and that memory — not FLOPs — decides how many conversations fit.

First, how much throughput does KV capacity actually buy?


In [ ]:
def measure_ceiling(kv_tokens, rate=60.0, n=800):
    reqs = sim_continuous(make_workload(n, rate), kv_tokens=kv_tokens)
    served = [r for r in reqs if r.done_at is not None]
    span = max(r.done_at for r in served) - min(r.arrival for r in reqs)
    return len(served) / span

avg_reserved = statistics.mean(r.prompt + r.output for r in _w)
print(f'{"KV budget (tokens)":>19} | {"≈ slots":>8} | {"ceiling req/s":>14} | gain')
print('-' * 58)
prev = None
for kv in (8_000, 15_000, 30_000, 60_000, 120_000, 240_000, 480_000):
    c = measure_ceiling(kv)
    gain = '' if prev is None else f'{c/prev:>6.2f}x'
    print(f'{kv:>19,} | {kv/avg_reserved:>8.0f} | {c:>14.2f} | {gain}')
    prev = c

print()
print('Doubling KV memory doubles throughput -- right up until it does not. Past ~120k')
print('tokens the curve flattens: the batch is now large enough that decode is compute-bound')
print('and prefill is eating the schedule. Buying more memory then buys nothing.')
print()
print('That flattening point is exactly the crossover from notebook 1, seen from the other')
print('side. Below it you are memory-limited and every KV byte you save is throughput.')
print('Above it you are compute-limited and KV optimisation is wasted engineering.')


### What eviction costs

The scheduler above cheated: it reserved `prompt + output` at admission, using an output
length it could not possibly know. Real schedulers must choose:

- **`reserve_max`** — book `max_tokens` for every request. Safe, never preempts, and wastes
  most of the reservation because the median reply is far shorter than the cap.
- **`optimistic`** — book only the prompt, admit aggressively, and **preempt** a sequence when
  the cache actually fills. Preemption means discarding that sequence's KV; when it resumes it
  must **re-prefill its prompt plus every token it had already generated**. This is
  recomputation, and it is pure waste.
- **`reserve_actual`** — the oracle. Impossible, included only as an upper bound.

There is a fourth option worth knowing: instead of discarding preempted KV, **swap it to CPU
memory** and copy it back later. That trades PCIe bandwidth for recompute — better when
contexts are long (recompute cost grows with context, copy cost grows more slowly).


In [ ]:
def sim_kv_policy(reqs, policy, kv_tokens=KV_TOKENS, max_batch=MAX_BATCH):
    """Continuous batching with an explicit, honest KV reservation policy."""
    clock = 0.0
    i, n = 0, len(reqs)
    active = []                       # [req, remaining, kv_actual, kv_reserved]
    waiting = deque()                 # [req, remaining, generated_so_far]
    reserved = 0
    preemptions = 0
    recomputed_tokens = 0

    while i < n or waiting or active:
        while i < n and reqs[i].arrival <= clock:
            waiting.append([reqs[i], reqs[i].output, 0]); i += 1
        if not active and not waiting:
            if i < n: clock = reqs[i].arrival; continue
            break

        admitted = 0
        while waiting and admitted < MAX_PREFILLS_PER_STEP and len(active) < max_batch:
            r, remaining, generated = waiting[0]
            prefill_len = r.prompt + generated            # resumed work re-reads what it made
            if   policy == 'reserve_max':    need = prefill_len + max(0, MAX_TOKENS - generated)
            elif policy == 'reserve_actual': need = prefill_len + remaining
            else:                            need = prefill_len          # optimistic
            if reserved + need > kv_tokens:
                break
            waiting.popleft(); admitted += 1
            clock += prefill_seconds(prefill_len)
            recomputed_tokens += generated                # the waste, counted honestly
            if r.first_token_at is None:
                r.first_token_at = clock
            reserved += need
            active.append([r, remaining, prefill_len, need])

        if not active:
            clock = max(clock, reqs[i].arrival) if i < n else clock + 0.001
            continue

        clock += decode_step_seconds(len(active))
        still = []
        for e in active:
            e[1] -= 1; e[2] += 1
            if policy == 'optimistic':
                e[3] += 1; reserved += 1                  # grows into memory it never booked
            if e[1] <= 0:
                e[0].done_at = clock; reserved -= e[3]
            else:
                still.append(e)
        active = still

        if policy == 'optimistic':                        # over-committed: something must go
            while reserved > kv_tokens and active:
                victim = active.pop()                     # evict the newest (least invested)
                reserved -= victim[3]
                preemptions += 1
                waiting.appendleft([victim[0], victim[1], victim[2] - victim[0].prompt])
    return reqs, preemptions, recomputed_tokens

print(f'{"policy":<16} {"thr req/s":>10} {"TTFT p50":>10} {"e2e p95":>9} '
      f'{"preempts":>9} {"wasted prefill":>15}')
print('-' * 76)
thr_by_policy = {}
for policy in ('reserve_max', 'optimistic', 'reserve_actual'):
    reqs = make_workload(1_500, 25.0)
    base_prefill = sum(r.prompt for r in reqs)
    rs, pre, rec = sim_kv_policy(reqs, policy)
    served = [r for r in rs if r.done_at is not None]
    e2e  = sorted(r.done_at - r.arrival for r in served)
    ttft = sorted(r.first_token_at - r.arrival for r in served)
    span = max(r.done_at for r in served) - min(r.arrival for r in rs)
    q = lambda a, p: a[min(len(a) - 1, int(p * len(a)))]
    thr_by_policy[policy] = len(served) / span
    print(f'{policy:<16} {thr_by_policy[policy]:>10.2f} {q(ttft,.5):>9.1f}s {q(e2e,.95):>8.1f}s '
          f'{pre:>9,} {rec/base_prefill:>14.1%}')

print()
print(f'reserve_max gives up {1 - thr_by_policy["reserve_max"]/thr_by_policy["optimistic"]:.0%} of the throughput that optimistic admission gets,')
print(f'and optimistic even edges past the oracle by {thr_by_policy["optimistic"]/thr_by_policy["reserve_actual"] - 1:.0%}. Meanwhile the recompute')
print('bill from all those preemptions is a rounding error on total prefill work.')


**The honest result, including the part that is inconvenient:**

- `reserve_max` — the "safe" policy — gives up **most of the replica's throughput**. It is
  paying for output tokens nobody generates. This is the hidden price of advertising a
  generous `max_tokens` default, and it is invisible in every dashboard you have.
- `optimistic` **beats even the oracle**. That is not a bug. Over-committing packs more
  sequences in during the window before the cache genuinely fills, and sequences finish at
  staggered times, so the over-commit usually resolves itself. Preemption is the price of a
  bet that mostly pays.
- The **recompute bill is a rounding error**, which is the surprise. Preemption tends to hit
  sequences late in life, and it only loses the tokens generated since admission. The real
  cost of preemption is not recompute — it is the **tail latency** of the users who get
  bounced, and the risk of thrashing at the memory boundary. Do not tune this by watching a
  "tokens recomputed" counter; watch p99.

⚖️ **What optimistic admission costs you:**

- **Livelock risk.** Evict-newest is stable; evict-oldest can starve a request forever as it
  is repeatedly bounced. Any preemption policy needs an anti-starvation rule.
- **The tail is much worse than the mean.** The users chosen as victims eat the whole cost.
  Average latency barely moves; p99 moves a lot. Measure the tail, not the mean.
- **You need paged KV to do this at all.** Preempting and re-admitting arbitrary sequences
  requires non-contiguous KV allocation (PagedAttention). With a naive contiguous allocator
  you get fragmentation and cannot use the memory you nominally freed.
- **It is unpredictable by construction.** The same request twice can take very different
  wall-clock time. If you have a hard per-request latency SLA, `reserve_max` and its lower
  ceiling may be the honest choice.


## ✅ Summary

- **Batch size is the master variable.** Decode at batch 1 runs a GPU at a few percent of its
  ability; the arithmetic-intensity calculation tells you the batch size you need, and it is
  in the hundreds.
- **Prefill and decode are different workloads.** Prefill is compute-bound (batching barely
  helps, long prompts are just expensive); decode is memory-bandwidth-bound (batching is the
  only lever, and it is a huge one).
- **Continuous batching beats static batching by ~2× and naive serving by ~40×**, because it
  never pays for padding and never blocks on the slowest member of a batch.
- **Streaming makes the response a long-lived connection**, which breaks idle timeouts,
  retries, and fast draining. Sequence-numbered frames and heartbeats are not garnish; they
  are what make the stream operable.
- **A bounded queue with fast rejection serves more users than an unbounded one.** Work done
  for a client who has left is worse than work not done at all.
- **KV cache capacity sets the throughput ceiling until compute takes over**, and the
  reservation policy for KV is a real throughput decision worth a third of your fleet.

Next: [Notebook 3 — Conversation State & Reliability](./03_conversation_state_and_reliability.ipynb).
